# 02-01 MySQL CRUD + JOIN

**JD 要求**: 精通至少一种数据库操作，如 Hive/MySQL 等

本节使用 **DuckDB**（内嵌，无需安装服务器）模拟 MySQL 语法进行学习。生产环境替换连接串即可，SQL 语法几乎完全兼容。

**本节目标**：
- DDL: CREATE TABLE, ALTER TABLE
- DML: INSERT, UPDATE, DELETE
- 查询: SELECT, WHERE, GROUP BY, ORDER BY, LIMIT
- JOIN: INNER / LEFT / RIGHT / FULL
- 子查询 & CTE

---

In [ ]:
import duckdb
import pandas as pd
import sys
sys.path.insert(0, "..")
from utils.data_generator import generate_ad_events

# 创建内存数据库
con = duckdb.connect()

# ============================================================
# DDL: 创建 B站广告平台的核心表
# ============================================================
con.execute("""
CREATE TABLE advertisers (
    advertiser_id VARCHAR PRIMARY KEY,
    name          VARCHAR NOT NULL,
    industry      VARCHAR,
    budget_daily  DECIMAL(10,2),
    created_at    TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
""")

con.execute("""
CREATE TABLE ads (
    ad_id         VARCHAR PRIMARY KEY,
    advertiser_id VARCHAR REFERENCES advertisers(advertiser_id),
    title         VARCHAR NOT NULL,
    ad_type       VARCHAR CHECK (ad_type IN ('image', 'video', 'text')),
    status        VARCHAR DEFAULT 'active',
    bid_price     DECIMAL(8,4)
);
""")

con.execute("""
CREATE TABLE ad_events (
    event_id      BIGINT,
    user_id       VARCHAR,
    ad_id         VARCHAR,
    advertiser_id VARCHAR,
    position      VARCHAR,
    action        VARCHAR,  -- impression / click / convert
    timestamp     TIMESTAMP,
    device        VARCHAR,
    cost          DECIMAL(8,4)
);
""")

print("表创建成功")

In [ ]:
# 插入测试数据
advertisers_data = [
    ("adv_1", "游戏公司A",  "游戏",   5000.00),
    ("adv_2", "美妆品牌B",  "美妆",  10000.00),
    ("adv_3", "教育平台C",  "教育",   3000.00),
    ("adv_4", "电商平台D",  "电商",  20000.00),
]
con.executemany("INSERT INTO advertisers VALUES (?, ?, ?, ?, CURRENT_TIMESTAMP)", advertisers_data)

ads_data = [
    ("ad_1", "adv_1", "新游戏皮肤上线",   "video", "active", 0.5),
    ("ad_2", "adv_1", "限时折扣装备",     "image", "active", 0.3),
    ("ad_3", "adv_2", "夏日美妆新品",     "image", "active", 0.8),
    ("ad_4", "adv_3", "在线编程课程",     "text",  "paused", 0.2),
    ("ad_5", "adv_4", "618大促活动",      "video", "active", 1.2),
]
con.executemany("INSERT INTO ads VALUES (?, ?, ?, ?, ?, ?)", ads_data)

# 从生成器导入事件数据
events = generate_ad_events(n=50000)
events_df = pd.DataFrame(events)
events_df['timestamp'] = pd.to_datetime(events_df['timestamp'])
con.execute("INSERT INTO ad_events SELECT * FROM events_df")
print(f"导入 {len(events_df):,} 条广告事件")

In [ ]:
# ============================================================
# 基础查询
# ============================================================
print("=== 广告投放商列表 ===")
con.sql("SELECT * FROM advertisers ORDER BY budget_daily DESC").show()

print("\n=== 活跃广告 ===")
con.sql("SELECT ad_id, title, ad_type, bid_price FROM ads WHERE status = 'active'").show()

In [ ]:
# ============================================================
# GROUP BY 聚合
# ============================================================
print("=== 各广告位点击率 CTR ===")
con.sql("""
SELECT 
    position,
    COUNT(*) FILTER (WHERE action = 'impression') AS impressions,
    COUNT(*) FILTER (WHERE action = 'click')      AS clicks,
    COUNT(*) FILTER (WHERE action = 'convert')    AS converts,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE action = 'click') 
             / NULLIF(COUNT(*) FILTER (WHERE action = 'impression'), 0), 
        2
    ) AS ctr_pct
FROM ad_events
GROUP BY position
ORDER BY ctr_pct DESC
""").show()

In [ ]:
# ============================================================
# JOIN：关联多表
# ============================================================
print("=== 各广告主的广告表现（INNER JOIN）===")
con.sql("""
SELECT 
    a.name         AS advertiser,
    a.industry,
    COUNT(DISTINCT e.ad_id)                                  AS ad_count,
    COUNT(*) FILTER (WHERE e.action = 'impression')          AS total_impressions,
    COUNT(*) FILTER (WHERE e.action = 'click')               AS total_clicks,
    ROUND(SUM(e.cost), 2)                                    AS total_cost,
    ROUND(AVG(e.cost) FILTER (WHERE e.action = 'click'), 4)  AS avg_cpc
FROM advertisers a
INNER JOIN ad_events e ON a.advertiser_id = e.advertiser_id
GROUP BY a.advertiser_id, a.name, a.industry
ORDER BY total_cost DESC
""").show()

In [ ]:
# LEFT JOIN：找出没有投放记录的广告
print("=== 未产生事件的广告（LEFT JOIN + IS NULL）===")
con.sql("""
SELECT ads.ad_id, ads.title, ads.status
FROM ads
LEFT JOIN ad_events ON ads.ad_id = ad_events.ad_id
WHERE ad_events.ad_id IS NULL
""").show()

In [ ]:
# ============================================================
# CTE（公共表表达式）—— 让复杂查询更可读
# ============================================================
print("=== 日均消耗 TOP3 广告主（CTE 写法）===")
con.sql("""
WITH daily_spend AS (
    -- 每天每广告主的消耗
    SELECT 
        advertiser_id,
        DATE_TRUNC('day', timestamp) AS day,
        SUM(cost) AS daily_cost
    FROM ad_events
    GROUP BY advertiser_id, DATE_TRUNC('day', timestamp)
),
avg_daily AS (
    -- 每广告主的平均日消耗
    SELECT advertiser_id, ROUND(AVG(daily_cost), 2) AS avg_daily_cost
    FROM daily_spend
    GROUP BY advertiser_id
)
SELECT a.name, ad.avg_daily_cost
FROM avg_daily ad
JOIN advertisers a USING (advertiser_id)
ORDER BY avg_daily_cost DESC
LIMIT 3
""").show()

## 自检题

1. 查询每种设备（device）的转化率（convert/impression）
2. 找出点击量 TOP 5 的广告，显示广告标题和广告主名称
3. 用 CTE 计算每个广告主的 ROI（假设 convert 产生 100 元收益）

**下一节**: `02_window_functions.ipynb` — 窗口函数（面试高频考点）